# PASSO 1 - SLM AUGMENTATION
#### Nesta etapa, vamos gerar samples novas utilizando os modelos Qwen 2.5 e 1.5B de parâmetros e GPT2 utilizando paráfrase. 

In [ ]:
import pandas as pd
import os
from pipeline.slm_augmenter import SLMAugmenter
from pipeline.preprocessor import Preprocessor

In [ ]:
raw_dataset = pd.read_csv('data/raw_datasets/mbti_1.csv')

Antes de tudo, vamos limpar tags html dos textos

In [ ]:
pre = Preprocessor('posts')

In [ ]:
no_html_dataset = pre.remove_html(raw_dataset)

In [ ]:
no_html_dataset.head()

### Opção A: Augmentation usando SLM local via Ollama (Qwen 2.5)

In [1]:
augmenter_qwen = SLMAugmenter(texts_column='posts', label_column='type', model_name='qwen2.5:1.5b')

NameError: name 'SLMAugmenter' is not defined

In [ ]:
df_final_qwen = augmenter_qwen.augment_minority_classes(
    df=no_html_dataset.copy(), 
    checkpoint_file='data/checkpoints/augmentation_checkpoint_qwen.csv',
    sample_size=5 
)

In [ ]:
df_final_qwen['augmented_posts'] = df_final_qwen['augmented_posts'].fillna(df_final_qwen['posts'])
df_final_qwen.head()

In [ ]:
os.makedirs('data/raw_datasets', exist_ok=True)
df_final_qwen.to_csv('data/raw_datasets/mbti_augmented_qwen.csv', index=False)

### Opção B: Augmentation usando Hugging Face Transformers (GPT-2)
O Ollama não estava disponível no ambiente que tentei rodar inicialmente. Logo, modifiquei para outro framework.

In [ ]:
import re
from transformers import pipeline, set_seed, AutoModelForCausalLM, AutoTokenizer

set_seed(42)

class HF_SLMAugmenter:
    def __init__(self, texts_column, label_column, model_name='gpt2'):
        self.texts_column = texts_column
        self.label_column = label_column
        self.model_name = model_name
        self.generator = None

        try:
            print(f"Loading '{self.model_name}' via Hugging Face...")
            # For GPT-2 we use CausalLM
            model = AutoModelForCausalLM.from_pretrained(self.model_name)
            tokenizer = AutoTokenizer.from_pretrained(self.model_name)

            # Setup device (GPU if available)
            device = 0 if os.environ.get('COLAB_GPU', '0') != '0' else -1
            self.generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=device)
            print(f"'{self.model_name}' loaded successfully.")
        except Exception as e:
            print(f"Error loading model: {e}. Trying generic pipeline...")
            try:
                self.generator = pipeline('text-generation', model=self.model_name)
            except:
                print("Could not initialize pipeline.")

    def _paraphrase_with_hf(self, original_text):
        if not self.generator:
            return original_text

        # Obs.: Perceba que aqui a técnica foi diferente, porque o GPT-2 funciona muito melhor completando textos do que respondendo a pedidos.
        prompt_input = f"Text: {original_text}\nParaphrase:"

        try:
            generated_output = self.generator(
                prompt_input,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_k=50,
                top_p=0.95,
                num_return_sequences=1,
                pad_token_id=self.generator.tokenizer.eos_token_id
            )

            full_text = generated_output[0]['generated_text']
   
            paraphrased_text = full_text.split("Paraphrase:")[-1].strip()

            if len(paraphrased_text.split()) < 3:
                return original_text

            return paraphrased_text

        except Exception as e:
            print(f"Error during generation: {e}")
            return None

    def augment_minority_classes(self, df, sample_size=None, checkpoint_file='augmentation_checkpoint.csv'):
        print(f"Starting augmentation using {self.model_name}...")

        if 'augmented_posts' not in df.columns:
            df['augmented_posts'] = pd.NA

        mask = df[self.label_column].str.contains('E|S', na=False)
        df_minoritario = df[mask]

        if sample_size:
            df_minoritario = df_minoritario.head(sample_size)

        dados_checkpoint = []
        indices_ja_processados = set()

        if checkpoint_file and os.path.exists(checkpoint_file):
            try:
                df_checkpoint = pd.read_csv(checkpoint_file)
                if 'original_index' in df_checkpoint.columns:
                    for _, row_cp in df_checkpoint.iterrows():
                        idx = row_cp['original_index']
                        df.at[idx, 'augmented_posts'] = row_cp['augmented_posts']
                        indices_ja_processados.add(idx)
                        dados_checkpoint.append({'original_index': idx, 'augmented_posts': row_cp['augmented_posts']})
            except:
                pass

        for index, row in df_minoritario.iterrows():
            if index in indices_ja_processados: continue

            texto_base = str(row[self.texts_column])
            frases = texto_base.split('|||')
            chunks = ['|||'.join(frases[i:i+3]) for i in range(0, len(frases), 3)]

            textos_gerados = []
            for chunk in chunks:
                if len(chunk.strip()) > 5:
                    res = self._paraphrase_with_hf(chunk)
                    if res: textos_gerados.append(res)

            if textos_gerados:
                texto_final = '|||'.join(textos_gerados)
                df.at[index, 'augmented_posts'] = texto_final
                dados_checkpoint.append({'original_index': index, 'augmented_posts': texto_final})
                print(f"Generated ({self.model_name}): {row[self.label_column]} at index {index}")

                if checkpoint_file:
                    os.makedirs(os.path.dirname(checkpoint_file), exist_ok=True)
                    pd.DataFrame(dados_checkpoint).to_csv(checkpoint_file, index=False)

        print("Augmentation finalizada!")
        return df

/Users/yasmimdantas/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/yasmimdantas/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

augmenter_gpt2 = HF_SLMAugmenter(
    texts_column="posts", label_column="type", model_name="gpt2"
)

df_final_gpt2 = augmenter_gpt2.augment_minority_classes(
    df=no_html_dataset.copy(),
    checkpoint_file='data/checkpoints/augmentation_checkpoint_gpt2.csv',
    sample_size=5 
)
df_final_gpt2['augmented_posts'] = df_final_gpt2['augmented_posts'].fillna(df_final_gpt2['posts'])
df_final_gpt2.head()

In [ ]:

df_final_gpt2.to_csv('data/raw_datasets/mbti_augmented_gpt2.csv', index=False)